In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import medmnist
from medmnist import PathMNIST
from sklearn.model_selection import train_test_split
import os

SEED = 42

# paths
PROJECT_DIR = '/path/to/thesis_project'
RAW_DIR     = os.path.join(PROJECT_DIR, 'data', 'raw')
PROC_DIR    = os.path.join(PROJECT_DIR, 'data', 'processed', f'seed_{SEED}')
os.makedirs(PROC_DIR, exist_ok=True)

print('Output directory:', PROC_DIR)

In [ ]:
#FASHIONMNIST

(ds_train, ds_test), ds_info = tfds.load("fashion_mnist",
                                         split = ["train", "test"],
                                         as_supervised = True,
                                         with_info = True,
                                         data_dir = os.path.join(RAW_DIR, "fashion_mnist")
                                        )

print(ds_info.features)

def tfds_to_numpy(ds):
    images, labels = [], []
    for img, lbl in ds:
        images.append(img.numpy())
        labels.append(int(lbl.numpy()))
    return np.array(images, dtype = np.uint8), np.array(labels, dtype = np.uint8)

X_trainval, y_trainval = tfds_to_numpy(ds_train)
X_test, y_test = tfds_to_numpy(ds_test)

print("train+val:", X_trainval.shape, y_trainval.shape)
print("test:", X_test.shape, y_test.shape)
print("dtype:", X_trainval.dtype)                                         

In [ ]:
#stratified 80/20 train/val split from train set

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size = 0.2,
    stratify = y_trainval,
    random_state = SEED
)

print("train:", X_train.shape, y_train.shape)
print("val:", X_val.shape, y_val.shape)
print("test:", X_test.shape, y_test.shape)

In [ ]:
class_names = ds_info.features["label"].names

print("{:<20} {:>8} {:>8} {:>8}".format("class", "train", "val", "test"))

for i, name in enumerate(class_names):
    print("{:<20} {:>8} {:>8} {:>8}".format(name, np.sum(y_train == i), np.sum(y_val == i), np.sum(y_test == i)))

In [ ]:
out_path = os.path.join(PROC_DIR, "fashion_mnist.npz")
np.savez(out_path,
         X_train = X_train, y_train = y_train,
         X_val = X_val, y_val = y_val,
         X_test = X_test, y_test = y_test,
         seed = np.array(SEED))

print("saved to:", out_path)

#sanity check

loaded = np.load(out_path)
print("\nKeys:", list(loaded.keys()))
for k in loaded:
    print("{}: shape = {}, dtype = {}, min = {}, max = {}".format(k, loaded[k].shape, loaded[k].dtype,
                                                                  loaded[k].min(), loaded[k].max()))

In [ ]:
#EUROSAT

data_dir = os.path.join(RAW_DIR, "EuroSAT_RGB")

ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    image_size = (64, 64),
    batch_size = None,
    shuffle = False
)

class_names = ds.class_names
print("classes:", class_names)

In [ ]:
images, labels = [], []

for img, lbl in ds:
    images.append(img.numpy().astype(np.uint8))
    labels.append(int(lbl.numpy()))

X_all = np.array(images, dtype = np.uint8)
y_all = np.array(labels, dtype = np.uint8)

print("all:", X_all.shape, y_all.shape)
print("dtype:", X_all.dtype)

In [ ]:
#stratified 70/15/15 train/val/test split from whole dataset

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all,
    test_size = 0.30,
    stratify = y_all,
    random_state = SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size = 0.50,
    stratify = y_temp,
    random_state = SEED
)

print("train:", X_train.shape, y_train.shape)
print("validation:", X_val.shape, y_val.shape)
print("test:", X_test.shape, y_test.shape)

In [ ]:
print("{:<20} {:>8} {:>8} {:>8}".format("class", "train", "val", "test"))

for i, name in enumerate(class_names):
    print("{:<20} {:>8} {:>8} {:>8}".format(name, np.sum(y_train == i), np.sum(y_val == i), np.sum(y_test == i)))

In [ ]:
out_path = os.path.join(PROC_DIR, "eurosat.npz")

np.savez(out_path,
         X_train = X_train,
         y_train = y_train,
         X_val = X_val,
         y_val = y_val,
         X_test = X_test,
         y_test = y_test,
         seed = np.array(SEED)
        )

print("Saved to:", out_path)

#sanity check

loaded = np.load(out_path)
print("\nKeys:", list(loaded.keys()))
for k in loaded:
    print("{}: shape = {}, dtype = {}, min = {}, max = {}".format(k, loaded[k].shape, loaded[k].dtype,
                                                                  loaded[k].min(), loaded[k].max()))

In [ ]:
#PATHMNIST

data_dir = os.path.join(RAW_DIR, "path_mnist")

ds_train = PathMNIST(split = "train", download = False, root = data_dir)
ds_val = PathMNIST(split = "val", download = False, root = data_dir)
ds_test = PathMNIST(split = "test", download = False, root = data_dir)

print('train:', len(ds_train))
print('val:  ', len(ds_val))
print('test: ', len(ds_test))

In [ ]:
def medmnist_to_numpy(ds):
    images, labels = [], []
    for img, lbl in ds:
        images.append(np.array(img, dtype = np.uint8))
        labels.append(int(lbl[0]))
    return np.array(images, dtype = np.uint8), np.array(labels, dtype = np.uint8)

X_train, y_train = medmnist_to_numpy(ds_train)
X_val, y_val = medmnist_to_numpy(ds_val)
X_test, y_test = medmnist_to_numpy(ds_test)

print("train:", X_train.shape, y_train.shape)
print("val:", X_val.shape, y_val.shape)
print("test:", X_test.shape, y_test.shape)
print("dtype:", X_train.dtype)

In [ ]:
print(ds_train.info, "\n")

class_names = ds_train.info["label"]
num_classes = len(class_names)

print("{:<40} {:>25} {:>25} {:>25}".format("Class", "Train", "Val", "Test"))

for i, name in enumerate(class_names.values()):
    print("{:<40} {:>25} {:>25} {:>25}".format(name, np.sum(y_train == i), np.sum(y_val == i), np.sum(y_test == i)))

In [ ]:
out_path = os.path.join(PROC_DIR, "path_mnist.npz")
np.savez(out_path,
         X_train = X_train, y_train = y_train,
         X_val = X_val, y_val = y_val,
         X_test = X_test, y_test = y_test
        )

print("Saved to:", out_path)

loaded = np.load(out_path)
print("\nKeys:", list(loaded.keys()))

for k in loaded:
    print(f"{k}: shape = {loaded[k].shape}, dtype = {loaded[k].dtype}, min = {loaded[k].min()}, max = {loaded[k].max()}")

In [ ]:
#SUMMARY

datasets = ["fashion_mnist.npz", "eurosat.npz", "path_mnist.npz"]

print("{:<20} {:<8} {:<20} {:<10} {:>5} {:>5}".format("Dataset", "split", "shape", "dtype", "min", "max"))
print("-"*75)

for file in datasets:
    data = np.load(os.path.join(PROC_DIR, file))
    for split in ["X_train", "X_val", "X_test"]:
        arr = data[split]
        print( "{:<20} {:<8} {:<20} {:<10} {:>5} {:>5}".format(file.replace(".npz", ""), split, str(arr.shape), str(arr.dtype),
        arr.min(), arr.max()) )
    print()